In [ ]:
!pip install pandas
!pip install chromadb
!pip install sentence-transformers
!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:
import pandas as pd

data = [
    ["Internet down for 3 days","Network Issue","P1"],
    ["Unable to login","Login Issue","P2"],
    ["Incorrect bill generated","Billing Issue","P2"],
]

df = pd.DataFrame(data, columns=["Complaint","Category","Priority"])

In [ ]:
def classify(text):
    text=text.lower()

    if "internet" in text:
        return "Network Issue"

    if "login" in text:
        return "Login Issue"

    if "bill" in text:
        return "Billing Issue"

    return "General"

In [ ]:
def priority(text):
    text=text.lower()

    if "3 days" in text or "business impact" in text:
        return "P1"

    return "P3"

In [ ]:
routing = {
    "Network Issue":"Network Team",
    "Login Issue":"IAM Team",
    "Billing Issue":"Finance Team"
}

In [ ]:
sla = {
    "P1":"4 Hours",
    "P2":"8 Hours",
    "P3":"24 Hours"
}

In [ ]:
def analyze_ticket(ticket):

    category = classify(ticket)

    pr = priority(ticket)

    team = routing.get(category)

    sla_time = sla.get(pr)

    return {
        "category": category,
        "priority": pr,
        "team": team,
        "sla": sla_time
    }

In [ ]:
analyze_ticket(
"Internet has been down for 3 days affecting business operations"
)

{'category': 'Network Issue',
 'priority': 'P1',
 'team': 'Network Team',
 'sla': '4 Hours'}

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis"
)

def get_sentiment(text):
    result = sentiment_model(text)[0]
    return result

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
get_sentiment(
"Internet has been down for 3 days and nobody is helping."
)

{'label': 'NEGATIVE', 'score': 0.9995179176330566}

In [ ]:
def escalation_risk(ticket):

    ticket = ticket.lower()

    score = 10

    if "3 days" in ticket:
        score += 30

    if "business impact" in ticket:
        score += 40

    if "urgent" in ticket:
        score += 20

    return min(score,100)

In [ ]:
def similar_cases(ticket):

    results = collection.query(
        query_texts=[ticket],
        n_results=2
    )

    return results

In [ ]:
def analyze(ticket):

    category = classify(ticket)

    pr = priority(ticket)

    team = routing[category]

    sentiment = get_sentiment(ticket)

    risk = escalation_risk(ticket)

    similar = similar_cases(ticket)

    return {
        "category": category,
        "priority": pr,
        "team": team,
        "sla": sla[pr],
        "sentiment": sentiment,
        "escalation_risk": risk,
        "similar_cases": similar
    }

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=analyze,
    inputs="text",
    outputs="json"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef5f8e5138e8b39713.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.launch(debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef5f8e5138e8b39713.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ef5f8e5138e8b39713.gradio.live


In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="tickets"
)

In [ ]:
collection.add(
    documents=[
        "Internet outage resolved by restarting router",
        "Billing issue resolved after invoice correction",
        "Login issue fixed after password reset"
    ],
    ids=["1","2","3"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.1MiB/s]


In [ ]:
collection.query(
    query_texts=["internet not working"],
    n_results=2
)

{'ids': [['1', '3']],
 'embeddings': None,
 'documents': [['Internet outage resolved by restarting router',
   'Login issue fixed after password reset']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.8429304361343384, 1.622509241104126]]}

In [ ]:
collection.query(
    query_texts=["internet not working"],
    n_results=2
)

{'ids': [['1', '3']],
 'embeddings': None,
 'documents': [['Internet outage resolved by restarting router',
   'Login issue fixed after password reset']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.8429304361343384, 1.622509241104126]]}

In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis"
)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
sentiment_model(
    "Internet has been down for 3 days and nobody is helping."
)

[{'label': 'NEGATIVE', 'score': 0.9995179176330566}]

In [ ]:
def escalation_risk(ticket):

    score = 0

    ticket = ticket.lower()

    if "3 days" in ticket:
        score += 30

    if "urgent" in ticket:
        score += 20

    if "business impact" in ticket:
        score += 40

    if "escalate" in ticket:
        score += 30

    return min(score,100)

In [ ]:
escalation_risk(
    "Internet down for 3 days causing business impact"
)

70

In [ ]:
def summarize(ticket):

    return ticket[:100]

In [ ]:
def analyze(ticket):

    category = classify(ticket)

    pr = priority(ticket)

    team = routing.get(
        category,
        "Support Team"
    )

    sentiment = sentiment_model(ticket)[0]

    risk = escalation_risk(ticket)

    similar = similar_cases(ticket)

    summary = summarize(ticket)

    return {
        "category": category,
        "priority": pr,
        "team": team,
        "sla": sla[pr],
        "sentiment": sentiment,
        "risk_score": risk,
        "summary": summary,
        "similar_cases": similar["documents"][0]
    }

In [ ]:
analyze(
"""
Internet has been down for 3 days.
Business operations affected.
Need urgent resolution.
"""
)

{'category': 'Network Issue',
 'priority': 'P1',
 'team': 'Network Team',
 'sla': '4 Hours',
 'sentiment': {'label': 'NEGATIVE', 'score': 0.9966865181922913},
 'risk_score': 50,
 'summary': '\nInternet has been down for 3 days.\nBusiness operations affected.\nNeed urgent resolution.\n',
 'similar_cases': ['Internet outage resolved by restarting router',
  'Billing issue resolved after invoice correction']}

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
!pip install -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 94.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
labels = [
    "Network Issue",
    "Billing Issue",
    "Login Issue"
]

result = classifier(
    "Internet has been down for 3 days",
    labels
)

print(result)

{'sequence': 'Internet has been down for 3 days', 'labels': ['Network Issue', 'Login Issue', 'Billing Issue'], 'scores': [0.6400359272956848, 0.27465295791625977, 0.08531105518341064]}


In [ ]:
all_tickets = []

In [ ]:
def analyze(ticket):

    result = {
        "ticket": ticket,
        "category": category,
        "priority": pr,
        "team": team,
        "sla": sla[pr]
    }

    all_tickets.append(result)

    return result

In [ ]:
from datetime import datetime

In [ ]:
def sla_status(priority):

    if priority == "P1":
        return "Critical"

    if priority == "P2":
        return "Warning"

    return "Healthy"

In [ ]:
import pandas as pd

def dashboard():

    return pd.DataFrame(all_tickets)

In [ ]:
dashboard()

,sequence,labels,scores
0,Internet has been down for 3 days,"[Network Issue, Login Issue, Billing Issue]","[0.6400359272956848, 0.27465295791625977, 0.08..."


In [ ]:
def metrics():

    total = len(all_tickets)

    critical = len([
        t for t in all_tickets
        if t["priority"] == "P1"
    ])

    return {
        "total_tickets": total,
        "critical_tickets": critical
    }

In [ ]:
def team_load():

    df = pd.DataFrame(all_tickets)

    return df["team"].value_counts()

In [ ]:
def category_stats():

    df = pd.DataFrame(all_tickets)

    return df["category"].value_counts()

In [ ]:
!pip install plotly

In [ ]:
print(all_tickets)

[{'sequence': 'Internet has been down for 3 days', 'labels': ['Network Issue', 'Login Issue', 'Billing Issue'], 'scores': [0.6400359272956848, 0.27465295791625977, 0.08531105518341064]}]


In [ ]:
all_tickets.append(result)


In [ ]:
print(all_tickets)

[{'sequence': 'Internet has been down for 3 days', 'labels': ['Network Issue', 'Login Issue', 'Billing Issue'], 'scores': [0.6400359272956848, 0.27465295791625977, 0.08531105518341064]}, {'sequence': 'Internet has been down for 3 days', 'labels': ['Network Issue', 'Login Issue', 'Billing Issue'], 'scores': [0.6400359272956848, 0.27465295791625977, 0.08531105518341064]}]


In [ ]:
analyze("Internet down for 3 days")
analyze("Unable to login portal")
analyze("Wrong bill generated")
analyze("Password reset required")
analyze("Business internet outage")

NameError: name 'category' is not defined

In [ ]:
def analyze(ticket):

    classification = classify(ticket)

    category = classification["category"]
    confidence = classification["confidence"]

    pr = priority(ticket)

    team = routing.get(
        category,
        "Support Team"
    )

    result = {
        "ticket": ticket,
        "category": category,
        "priority": pr,
        "team": team,
        "sla": sla[pr]
    }

    all_tickets.append(result)

    return result

In [ ]:
analyze("Internet down for 3 days")

NameError: name 'classify' is not defined

In [ ]:
labels = [
    "Network Issue",
    "Billing Issue",
    "Login Issue"
]

def classify(ticket):

    result = classifier(
        ticket,
        labels
    )

    return {
        "category": result["labels"][0],
        "confidence": round(
            result["scores"][0] * 100,
            2
        )
    }

In [ ]:
classify("Internet down for 3 days")

{'category': 'Network Issue', 'confidence': 74.7}

In [ ]:
analyze("Internet down for 3 days")


NameError: name 'routing' is not defined

In [ ]:
routing = {
    "Network Issue": "Network Team",
    "Login Issue": "IAM Team",
    "Billing Issue": "Finance Team"
}

In [ ]:
def priority(ticket):

    ticket = ticket.lower()

    if "3 days" in ticket:
        return "P1"

    if "business" in ticket:
        return "P1"

    if "urgent" in ticket:
        return "P1"

    return "P3"

In [ ]:
sla = {
    "P1": "4 Hours",
    "P2": "8 Hours",
    "P3": "24 Hours"
}

In [ ]:
all_tickets = []

In [ ]:
print(routing)
print(sla)
print(all_tickets)

{'Network Issue': 'Network Team', 'Login Issue': 'IAM Team', 'Billing Issue': 'Finance Team'}
{'P1': '4 Hours', 'P2': '8 Hours', 'P3': '24 Hours'}
[]


In [ ]:
analyze("Internet down for 3 days")

{'ticket': 'Internet down for 3 days',
 'category': 'Network Issue',
 'priority': 'P1',
 'team': 'Network Team',
 'sla': '4 Hours'}

In [ ]:
analyze("Internet down for 3 days")
analyze("Unable to login portal")
analyze("Wrong bill generated")
analyze("Password reset required")
analyze("Business internet outage")

{'ticket': 'Business internet outage',
 'category': 'Network Issue',
 'priority': 'P1',
 'team': 'Network Team',
 'sla': '4 Hours'}

In [ ]:
dashboard()

,ticket,category,priority,team,sla
0,Internet down for 3 days,Network Issue,P1,Network Team,4 Hours
1,Internet down for 3 days,Network Issue,P1,Network Team,4 Hours
2,Unable to login portal,Login Issue,P3,IAM Team,24 Hours
3,Wrong bill generated,Billing Issue,P3,Finance Team,24 Hours
4,Password reset required,Login Issue,P3,IAM Team,24 Hours
5,Business internet outage,Network Issue,P1,Network Team,4 Hours


In [ ]:
print(len(all_tickets))

6


In [ ]:
classification = classify(ticket)

category = classification["category"]
confidence = classification["confidence"]

NameError: name 'ticket' is not defined

In [ ]:
import pandas as pd

df = pd.DataFrame(all_tickets)

print("Total Tickets:", len(df))
print("Critical Tickets:",
      len(df[df["priority"]=="P1"]))

print(df["team"].value_counts())

Total Tickets: 6
Critical Tickets: 3
team
Network Team    3
IAM Team        2
Finance Team    1
Name: count, dtype: int64


In [ ]:
import gradio as gr
import pandas as pd

def dashboard_view(ticket):

    result = analyze(ticket)

    df = pd.DataFrame(all_tickets)

    total_tickets = len(df)

    critical_tickets = len(
        df[df["priority"]=="P1"]
    )

    team_stats = (
        df["team"]
        .value_counts()
        .to_string()
    )

    category_stats = (
        df["category"]
        .value_counts()
        .to_string()
    )

    dashboard_summary = f"""
Total Tickets: {total_tickets}

Critical Tickets: {critical_tickets}

Team Workload:
{team_stats}

Category Distribution:
{category_stats}
"""

    return result, dashboard_summary, df


with gr.Blocks() as demo:

    gr.Markdown(
        "# 🚀 AI Customer Complaint Routing Engine"
    )

    with gr.Row():
        ticket_input = gr.Textbox(
            label="Customer Complaint",
            lines=4
        )

    submit_btn = gr.Button(
        "Analyze Ticket"
    )

    result_output = gr.JSON(
        label="AI Analysis"
    )

    dashboard_output = gr.Textbox(
        label="Dashboard Metrics"
    )

    table_output = gr.Dataframe(
        label="Live Ticket Stream"
    )

    submit_btn.click(
        fn=dashboard_view,
        inputs=ticket_input,
        outputs=[
            result_output,
            dashboard_output,
            table_output
        ]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8802f80b4a93662be7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8802f80b4a93662be7.gradio.live


In [ ]:
labels = [
    "Network Issue",
    "Billing Issue",
    "Login Issue",
    "Fraud Issue",
    "Account Security Issue",
    "Card Issue",
    "Technical Support",
    "Service Request"
]

In [ ]:
def priority(ticket):

    ticket = ticket.lower()

    critical_keywords = [
        "credit card stolen",
        "fraud",
        "hacked",
        "unauthorized transaction",
        "account compromised",
        "security breach",
        "business impact",
        "service outage"
    ]

    for word in critical_keywords:
        if word in ticket:
            return "P1"

    high_keywords = [
        "payment failed",
        "money deducted",
        "cannot login",
        "internet down"
    ]

    for word in high_keywords:
        if word in ticket:
            return "P2"

    return "P3"

In [ ]:
routing = {
    "Network Issue":"Network Team",
    "Billing Issue":"Finance Team",
    "Login Issue":"IAM Team",
    "Fraud Issue":"Fraud Investigation Team",
    "Account Security Issue":"Cyber Security Team",
    "Card Issue":"Card Operations Team",
    "Technical Support":"Support Team",
    "Service Request":"Service Desk"
}

In [ ]:
!pip install transformers accelerate sentencepiece

In [ ]:
from transformers import pipeline



In [ ]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=128
)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
from transformers.pipelines import SUPPORTED_TASKS

print(SUPPORTED_TASKS.keys())

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'table-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'keypoint-matching', 'any-to-any'])


In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

In [ ]:
response = client.chat.completions.create(
    model="meta-llama/llama-3.3-8b-instruct:free",
    messages=[
        {"role":"user",
         "content":"Credit card stolen and unauthorized transactions detected."}
    ]
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found for meta-llama/llama-3.3-8b-instruct:free.', 'code': 404}, 'user_id': 'user_3ErZ52YKp1K8xhz7oSLgVAjWE9z'}

In [ ]:
priority_labels = [
    "P1 Critical",
    "P2 High",
    "P3 Medium"
]

def predict_priority(ticket):
    result = classifier(
        ticket,
        priority_labels
    )

    return result["labels"][0]

In [ ]:
predict_priority(
    "Credit card stolen and unauthorized transactions detected"
)

'P3 Medium'

In [ ]:
import pandas as pd

data = [
["Internet down for 3 days affecting business operations","Network Issue","P1","Network Team"],
["Credit card stolen and unauthorized transactions detected","Fraud Issue","P1","Fraud Team"],
["Unable to login to internet banking","Login Issue","P2","IAM Team"],
["Money deducted but payment failed","Billing Issue","P2","Finance Team"],
["Need new debit card issued","Service Request","P3","Service Desk"]
]

df = pd.DataFrame(
    data,
    columns=["Complaint","Category","Priority","Team"]
)

df.to_csv("complaints.csv",index=False)

In [ ]:
prompt = """
Generate 100 banking customer complaints.

Output CSV format:

Complaint,Category,Priority,Team

Categories:
Network Issue
Billing Issue
Login Issue
Fraud Issue
Account Security Issue
Service Request

Use realistic complaints.
"""

In [ ]:
all_rows.extend(generated_rows)

NameError: name 'all_rows' is not defined

In [ ]:
csv_content = """
Complaint,Category,Priority,Team
Internet banking unavailable for 4 hours,Network Issue,P1,Network Team
Branch network outage affecting transactions,Network Issue,P1,Network Team
Mobile banking app not loading,Network Issue,P2,Network Team
Website inaccessible since morning,Network Issue,P2,Network Team
VPN connection failing for employees,Network Issue,P2,Network Team
Slow internet banking response time,Network Issue,P3,Network Team
Unable to access account dashboard,Network Issue,P2,Network Team
ATM network connectivity issue,Network Issue,P1,Network Team
Payment gateway timing out,Network Issue,P2,Network Team
Frequent disconnection during online banking,Network Issue,P3,Network Team
Corporate banking portal unavailable,Network Issue,P1,Network Team
Transaction history page not loading,Network Issue,P3,Network Team
Online transfer page crashing repeatedly,Network Issue,P2,Network Team
Network error while making payment,Network Issue,P2,Network Team
Banking services unavailable due to server issue,Network Issue,P1,Network Team

Money deducted but payment failed,Billing Issue,P2,Finance Team
Duplicate charge on my account,Billing Issue,P2,Finance Team
Refund not received after cancellation,Billing Issue,P2,Finance Team
Incorrect credit card bill generated,Billing Issue,P2,Finance Team
Unexpected service charges applied,Billing Issue,P3,Finance Team
EMI charged twice this month,Billing Issue,P2,Finance Team
Invoice amount mismatch,Billing Issue,P3,Finance Team
Wrong GST amount on statement,Billing Issue,P3,Finance Team
Account debited without successful transaction,Billing Issue,P2,Finance Team
Loan payment reflected incorrectly,Billing Issue,P2,Finance Team
Interest calculation seems incorrect,Billing Issue,P3,Finance Team
Credit card annual fee charged unexpectedly,Billing Issue,P3,Finance Team
Auto-debit executed twice,Billing Issue,P2,Finance Team
Balance deduction without explanation,Billing Issue,P2,Finance Team
Cashback not credited to account,Billing Issue,P3,Finance Team

Unable to login to internet banking,Login Issue,P2,IAM Team
OTP not received during login,Login Issue,P2,IAM Team
Account locked after failed attempts,Login Issue,P2,IAM Team
Password reset required,Login Issue,P3,IAM Team
Login page displaying error message,Login Issue,P2,IAM Team
Face authentication not working,Login Issue,P2,IAM Team
Username not recognized by system,Login Issue,P2,IAM Team
Login session expires immediately,Login Issue,P3,IAM Team
Two-factor authentication failing,Login Issue,P2,IAM Team
Corporate account login inaccessible,Login Issue,P2,IAM Team
Unable to sign in using mobile app,Login Issue,P2,IAM Team
Password reset link not received,Login Issue,P3,IAM Team
Authentication service unavailable,Login Issue,P2,IAM Team
Security question verification failing,Login Issue,P3,IAM Team
Customer portal login not working,Login Issue,P2,IAM Team

Credit card stolen and unauthorized transactions detected,Fraud Issue,P1,Fraud Team
Unknown charges appearing on account,Fraud Issue,P1,Fraud Team
Fraudulent online purchase detected,Fraud Issue,P1,Fraud Team
Unauthorized transfer from savings account,Fraud Issue,P1,Fraud Team
Debit card used without permission,Fraud Issue,P1,Fraud Team
Suspicious withdrawal from ATM,Fraud Issue,P1,Fraud Team
International transaction not initiated by me,Fraud Issue,P1,Fraud Team
Fake merchant charged my card,Fraud Issue,P1,Fraud Team
Fraudulent UPI transaction detected,Fraud Issue,P1,Fraud Team
Multiple unauthorized purchases observed,Fraud Issue,P1,Fraud Team
Card details compromised after online purchase,Fraud Issue,P1,Fraud Team
Unknown beneficiary added and funds transferred,Fraud Issue,P1,Fraud Team
Fraud alert received for my account,Fraud Issue,P1,Fraud Team
Suspicious recurring charges detected,Fraud Issue,P1,Fraud Team
Account balance reduced by unauthorized activity,Fraud Issue,P1,Fraud Team

Account hacked and email changed,Account Security Issue,P1,Cyber Security Team
Suspicious login from another country,Account Security Issue,P1,Cyber Security Team
Password changed without authorization,Account Security Issue,P1,Cyber Security Team
Multiple failed login attempts detected,Account Security Issue,P2,Cyber Security Team
Received security alert for unknown device,Account Security Issue,P1,Cyber Security Team
Unauthorized profile changes observed,Account Security Issue,P1,Cyber Security Team
MFA disabled without my knowledge,Account Security Issue,P1,Cyber Security Team
Security questions modified unexpectedly,Account Security Issue,P1,Cyber Security Team
Unknown browser accessed my account,Account Security Issue,P1,Cyber Security Team
Account recovery information altered,Account Security Issue,P1,Cyber Security Team
Unauthorized device linked to account,Account Security Issue,P1,Cyber Security Team
Repeated brute-force login attempts detected,Account Security Issue,P2,Cyber Security Team
Security notification received unexpectedly,Account Security Issue,P2,Cyber Security Team
Email alerts stopped after account changes,Account Security Issue,P1,Cyber Security Team
Account access granted to unknown user,Account Security Issue,P1,Cyber Security Team

Need new debit card issued,Service Request,P3,Service Desk
Request account statement for last year,Service Request,P3,Service Desk
Update registered mobile number,Service Request,P3,Service Desk
Request cheque book delivery,Service Request,P3,Service Desk
Need account closure procedure,Service Request,P3,Service Desk
Request address update on account,Service Request,P3,Service Desk
Apply for new credit card,Service Request,P3,Service Desk
Need loan foreclosure certificate,Service Request,P3,Service Desk
Request nominee addition to account,Service Request,P3,Service Desk
Need bank balance certificate,Service Request,P3,Service Desk
Request debit card PIN regeneration,Service Request,P3,Service Desk
Apply for internet banking access,Service Request,P3,Service Desk
Need transaction statement PDF,Service Request,P3,Service Desk
Request upgrade to premium account,Service Request,P3,Service Desk
Need KYC update assistance,Service Request,P3,Service Desk

Printer not working in branch office,Technical Support,P3,Support Team
ATM receipt printer malfunctioning,Technical Support,P2,Support Team
Scanner unable to upload documents,Technical Support,P3,Support Team
Employee workstation freezing frequently,Technical Support,P3,Support Team
Desktop application crashing on startup,Technical Support,P2,Support Team
Card reader not detecting cards,Technical Support,P2,Support Team
POS machine not responding,Technical Support,P2,Support Team
Branch server showing hardware error,Technical Support,P1,Support Team
Document upload feature broken,Technical Support,P2,Support Team
Customer kiosk touchscreen unresponsive,Technical Support,P2,Support Team
Email client not syncing messages,Technical Support,P3,Support Team
System update failed on workstation,Technical Support,P3,Support Team
USB token not recognized by system,Technical Support,P2,Support Team
Branch printer offline unexpectedly,Technical Support,P3,Support Team
Hardware failure affecting branch operations,Technical Support,P1,Support Team
"""

with open("complaints.csv", "w", encoding="utf-8") as f:
    f.write(csv_content)

print("complaints.csv created successfully")

complaints.csv created successfully


In [ ]:
import pandas as pd

df = pd.read_csv("generated_text.csv")

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'generated_text.csv'

In [ ]:
import pandas as pd
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

prompt = """
Generate 200 unique banking complaints in CSV format.

Columns:
Complaint,Category,Priority,Team

Categories:
Network Issue, Billing Issue, Login Issue,
Fraud Issue, Account Security Issue,
Service Request, Technical Support

Return only CSV rows.
"""

result = generator(
    prompt,
    max_new_tokens=4000,
    do_sample=True,
    temperature=0.9
)

print(result[0]["generated_text"])

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
import pandas as pd
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

prompt = """
Generate 200 unique banking complaints in CSV format.

Columns:
Complaint,Category,Priority,Team

Categories:
Network Issue, Billing Issue, Login Issue,
Fraud Issue, Account Security Issue,
Service Request, Technical Support

Return only CSV rows.
"""

result = generator(
    prompt,
    max_new_tokens=4000,
    do_sample=True,
    temperature=0.9
)

print(result[0]["generated_text"])

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausal


Generate 200 unique banking complaints in CSV format.

Columns:
Complaint,Category,Priority,Team

Categories:
Network Issue, Billing Issue, Login Issue,
Fraud Issue, Account Security Issue,
Service Request, Technical Support

Return only CSV rows.
'Login Issue', 'Fraud Issue', 'Account Security Issue', 'Service Request', 'Technical Support', 'Service Request', 'Network Issue', 'Hazard Issue', 'Network Issue', 'Login Issue', 'Verify', 'Login Issue', 'Verify', 'Tenant Issue', 'Shipper', 'Receiver', 'Service Request', 'Record Security Issue', 'Hazard Issue', 'Shipper', 'Verify', 'Technical Support', 'Hazard Issue', 'Receiver', 'Importer', 'Receiver', 'Receiver', 'Authority Issue', 'Insurance Issue', 'Verify', 'Attack Security Issue', 'Record Security Issue', 'Importer', 'Sales', 'Technical Support', 'Receiver', 'Receiver', 'Tenant Issue', 'Contract Security Issue', 'Service Request', 'Receiver', 'Tenant Issue', 'Format Security Issue', 'Password Issue', 'Credit Issue', 'Telephone Issue',

In [ ]:
import pandas as pd

df = pd.read_csv("complaints.csv")

print(df.shape)
print(df.head())

(105, 4)
                                      Complaint       Category Priority  \
0      Internet banking unavailable for 4 hours  Network Issue       P1   
1  Branch network outage affecting transactions  Network Issue       P1   
2                Mobile banking app not loading  Network Issue       P2   
3            Website inaccessible since morning  Network Issue       P2   
4          VPN connection failing for employees  Network Issue       P2   

           Team  
0  Network Team  
1  Network Team  
2  Network Team  
3  Network Team  
4  Network Team  


In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
label_map = {
    "Network Issue":0,
    "Billing Issue":1,
    "Login Issue":2,
    "Fraud Issue":3,
    "Account Security Issue":4,
    "Service Request":5,
    "Technical Support":6
}

df["label"] = df["Category"].map(label_map)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["Complaint"],
        truncation=True,
        padding="max_length"
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [ ]:
print(train_ds)
print(test_ds)

Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 84
})
Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 21
})


In [ ]:
print(train_ds)
print(test_ds)

Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 84
})
Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 21
})


In [ ]:
train_ds = train_ds.remove_columns(
    ["Complaint","Category","Priority","Team"]
)

test_ds = test_ds.remove_columns(
    ["Complaint","Category","Priority","Team"]
)

train_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

---------------------------------------------------------------------------
NOTE: If your import is failing due to a missing package, you can
manually install dependencies using either !pip or !apt.

To view examples of installing some common dependencies, click the
"Open Examples" button below.
---------------------------------------------------------------------------

SyntaxError: invalid syntax (2704702290.py, line 1)

In [ ]:
!pip uninstall -y torchvision torch torchaudio
!pip install torch torchvision torchaudio
!pip install -U transformers datasets accelerate evaluate

Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 634.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch

print(torch.cuda.is_available())


False


In [ ]:
import torch
import torchvision
import transformers

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Transformers:", transformers.__version__)

Torch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
Transformers: 5.9.0


In [ ]:
import torch

print(torch.cuda.is_available())

True


In [ ]:
print(torch.__version__)
print(torchvision.__version__)
print(transformers.__version__)

2.11.0+cu128
0.26.0+cu128
5.9.0


In [ ]:
from transformers import Trainer

print("Trainer imported successfully")

Trainer imported successfully


In [ ]:
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
import pandas as pd

df = pd.read_csv("complaints.csv")
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'complaints.csv'

In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("complaints.csv")

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'complaints.csv'

In [ ]:
csv_content = """
Complaint,Category,Priority,Team
Internet banking unavailable for 4 hours,Network Issue,P1,Network Team
Branch network outage affecting transactions,Network Issue,P1,Network Team
Mobile banking app not loading,Network Issue,P2,Network Team
Website inaccessible since morning,Network Issue,P2,Network Team
VPN connection failing for employees,Network Issue,P2,Network Team
Slow internet banking response time,Network Issue,P3,Network Team
Unable to access account dashboard,Network Issue,P2,Network Team
ATM network connectivity issue,Network Issue,P1,Network Team
Payment gateway timing out,Network Issue,P2,Network Team
Frequent disconnection during online banking,Network Issue,P3,Network Team
Corporate banking portal unavailable,Network Issue,P1,Network Team
Transaction history page not loading,Network Issue,P3,Network Team
Online transfer page crashing repeatedly,Network Issue,P2,Network Team
Network error while making payment,Network Issue,P2,Network Team
Banking services unavailable due to server issue,Network Issue,P1,Network Team

Money deducted but payment failed,Billing Issue,P2,Finance Team
Duplicate charge on my account,Billing Issue,P2,Finance Team
Refund not received after cancellation,Billing Issue,P2,Finance Team
Incorrect credit card bill generated,Billing Issue,P2,Finance Team
Unexpected service charges applied,Billing Issue,P3,Finance Team
EMI charged twice this month,Billing Issue,P2,Finance Team
Invoice amount mismatch,Billing Issue,P3,Finance Team
Wrong GST amount on statement,Billing Issue,P3,Finance Team
Account debited without successful transaction,Billing Issue,P2,Finance Team
Loan payment reflected incorrectly,Billing Issue,P2,Finance Team
Interest calculation seems incorrect,Billing Issue,P3,Finance Team
Credit card annual fee charged unexpectedly,Billing Issue,P3,Finance Team
Auto-debit executed twice,Billing Issue,P2,Finance Team
Balance deduction without explanation,Billing Issue,P2,Finance Team
Cashback not credited to account,Billing Issue,P3,Finance Team

Unable to login to internet banking,Login Issue,P2,IAM Team
OTP not received during login,Login Issue,P2,IAM Team
Account locked after failed attempts,Login Issue,P2,IAM Team
Password reset required,Login Issue,P3,IAM Team
Login page displaying error message,Login Issue,P2,IAM Team
Face authentication not working,Login Issue,P2,IAM Team
Username not recognized by system,Login Issue,P2,IAM Team
Login session expires immediately,Login Issue,P3,IAM Team
Two-factor authentication failing,Login Issue,P2,IAM Team
Corporate account login inaccessible,Login Issue,P2,IAM Team
Unable to sign in using mobile app,Login Issue,P2,IAM Team
Password reset link not received,Login Issue,P3,IAM Team
Authentication service unavailable,Login Issue,P2,IAM Team
Security question verification failing,Login Issue,P3,IAM Team
Customer portal login not working,Login Issue,P2,IAM Team

Credit card stolen and unauthorized transactions detected,Fraud Issue,P1,Fraud Team
Unknown charges appearing on account,Fraud Issue,P1,Fraud Team
Fraudulent online purchase detected,Fraud Issue,P1,Fraud Team
Unauthorized transfer from savings account,Fraud Issue,P1,Fraud Team
Debit card used without permission,Fraud Issue,P1,Fraud Team
Suspicious withdrawal from ATM,Fraud Issue,P1,Fraud Team
International transaction not initiated by me,Fraud Issue,P1,Fraud Team
Fake merchant charged my card,Fraud Issue,P1,Fraud Team
Fraudulent UPI transaction detected,Fraud Issue,P1,Fraud Team
Multiple unauthorized purchases observed,Fraud Issue,P1,Fraud Team
Card details compromised after online purchase,Fraud Issue,P1,Fraud Team
Unknown beneficiary added and funds transferred,Fraud Issue,P1,Fraud Team
Fraud alert received for my account,Fraud Issue,P1,Fraud Team
Suspicious recurring charges detected,Fraud Issue,P1,Fraud Team
Account balance reduced by unauthorized activity,Fraud Issue,P1,Fraud Team

Account hacked and email changed,Account Security Issue,P1,Cyber Security Team
Suspicious login from another country,Account Security Issue,P1,Cyber Security Team
Password changed without authorization,Account Security Issue,P1,Cyber Security Team
Multiple failed login attempts detected,Account Security Issue,P2,Cyber Security Team
Received security alert for unknown device,Account Security Issue,P1,Cyber Security Team
Unauthorized profile changes observed,Account Security Issue,P1,Cyber Security Team
MFA disabled without my knowledge,Account Security Issue,P1,Cyber Security Team
Security questions modified unexpectedly,Account Security Issue,P1,Cyber Security Team
Unknown browser accessed my account,Account Security Issue,P1,Cyber Security Team
Account recovery information altered,Account Security Issue,P1,Cyber Security Team
Unauthorized device linked to account,Account Security Issue,P1,Cyber Security Team
Repeated brute-force login attempts detected,Account Security Issue,P2,Cyber Security Team
Security notification received unexpectedly,Account Security Issue,P2,Cyber Security Team
Email alerts stopped after account changes,Account Security Issue,P1,Cyber Security Team
Account access granted to unknown user,Account Security Issue,P1,Cyber Security Team

Need new debit card issued,Service Request,P3,Service Desk
Request account statement for last year,Service Request,P3,Service Desk
Update registered mobile number,Service Request,P3,Service Desk
Request cheque book delivery,Service Request,P3,Service Desk
Need account closure procedure,Service Request,P3,Service Desk
Request address update on account,Service Request,P3,Service Desk
Apply for new credit card,Service Request,P3,Service Desk
Need loan foreclosure certificate,Service Request,P3,Service Desk
Request nominee addition to account,Service Request,P3,Service Desk
Need bank balance certificate,Service Request,P3,Service Desk
Request debit card PIN regeneration,Service Request,P3,Service Desk
Apply for internet banking access,Service Request,P3,Service Desk
Need transaction statement PDF,Service Request,P3,Service Desk
Request upgrade to premium account,Service Request,P3,Service Desk
Need KYC update assistance,Service Request,P3,Service Desk

Printer not working in branch office,Technical Support,P3,Support Team
ATM receipt printer malfunctioning,Technical Support,P2,Support Team
Scanner unable to upload documents,Technical Support,P3,Support Team
Employee workstation freezing frequently,Technical Support,P3,Support Team
Desktop application crashing on startup,Technical Support,P2,Support Team
Card reader not detecting cards,Technical Support,P2,Support Team
POS machine not responding,Technical Support,P2,Support Team
Branch server showing hardware error,Technical Support,P1,Support Team
Document upload feature broken,Technical Support,P2,Support Team
Customer kiosk touchscreen unresponsive,Technical Support,P2,Support Team
Email client not syncing messages,Technical Support,P3,Support Team
System update failed on workstation,Technical Support,P3,Support Team
USB token not recognized by system,Technical Support,P2,Support Team
Branch printer offline unexpectedly,Technical Support,P3,Support Team
Hardware failure affecting branch operations,Technical Support,P1,Support Team
"""

with open("complaints.csv", "w", encoding="utf-8") as f:
    f.write(csv_content)

print("complaints.csv created successfully")

complaints.csv created successfully


In [ ]:
import pandas as pd

df = pd.read_csv("complaints.csv")

print(df.shape)
df.head()

(105, 4)


,Complaint,Category,Priority,Team
0,Internet banking unavailable for 4 hours,Network Issue,P1,Network Team
1,Branch network outage affecting transactions,Network Issue,P1,Network Team
2,Mobile banking app not loading,Network Issue,P2,Network Team
3,Website inaccessible since morning,Network Issue,P2,Network Team
4,VPN connection failing for employees,Network Issue,P2,Network Team


In [ ]:
label_map = {
    "Network Issue":0,
    "Billing Issue":1,
    "Login Issue":2,
    "Fraud Issue":3,
    "Account Security Issue":4,
    "Service Request":5,
    "Technical Support":6
}

df["label"] = df["Category"].map(label_map)

df.head()

,Complaint,Category,Priority,Team,label
0,Internet banking unavailable for 4 hours,Network Issue,P1,Network Team,0
1,Branch network outage affecting transactions,Network Issue,P1,Network Team,0
2,Mobile banking app not loading,Network Issue,P2,Network Team,0
3,Website inaccessible since morning,Network Issue,P2,Network Team,0
4,VPN connection failing for employees,Network Issue,P2,Network Team,0


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

print(train_ds)
print(test_ds)

Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__'],
    num_rows: 84
})
Dataset({
    features: ['Complaint', 'Category', 'Priority', 'Team', 'label', '__index_level_0__'],
    num_rows: 21
})


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(batch):

    return tokenizer(
        batch["Complaint"],
        truncation=True,
        padding="max_length"
    )

In [ ]:
train_ds = train_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [ ]:
train_ds = train_ds.remove_columns(
    ["Complaint","Category","Priority","Team"]
)

test_ds = test_ds.remove_columns(
    ["Complaint","Category","Priority","Team"]
)

In [ ]:
train_ds = train_ds.rename_column(
    "label",
    "labels"
)

test_ds = test_ds.rename_column(
    "label",
    "labels"
)

In [ ]:
train_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load(
    "accuracy"
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline
import gradio as gr

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

categories = [
    "Network Issue",
    "Billing Issue",
    "Login Issue",
    "Fraud Issue",
    "Account Security Issue",
    "Service Request",
    "Technical Support"
]

priorities = [
    "P1 Critical",
    "P2 High",
    "P3 Medium"
]

def analyze_complaint(ticket):

    category_result = classifier(
        ticket,
        categories
    )

    priority_result = classifier(
        ticket,
        priorities
    )

    category = category_result["labels"][0]
    category_conf = round(
        category_result["scores"][0] * 100,
        2
    )

    priority = priority_result["labels"][0]

    routing = {
        "Network Issue":"Network Team",
        "Billing Issue":"Finance Team",
        "Login Issue":"IAM Team",
        "Fraud Issue":"Fraud Team",
        "Account Security Issue":"Cyber Security Team",
        "Service Request":"Service Desk",
        "Technical Support":"Support Team"
    }

    team = routing.get(
        category,
        "Support Team"
    )

    return {
        "Category": category,
        "Confidence": f"{category_conf}%",
        "Priority": priority,
        "Assigned Team": team
    }

demo = gr.Interface(
    fn=analyze_complaint,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter customer complaint..."
    ),
    outputs="json",
    title="AI Customer Complaint Routing Engine",
    description="Classify, prioritize and route customer complaints using BART."
)

demo.launch()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e1d46d479536bd740.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
